# GMAI-Pulse — CoverMe URL Scope Inventory

**Purpose.** Answer the question no run has answered yet in full: *what URLs actually
exist for the CoverMe product suite?* This notebook scans the single-suite CoverMe
table `csdo_prod_catalog.adobe_coverme_bronze.hit_data` over full history with **no
URL filter applied**, and lands the complete distinct-URL inventory in Delta plus a
scannable rollup.

**Why.** Every scope decision so far inherits `databricks/conf/settings.py` (`broad`
= `%coverme.com%`, `%pourmeproteger.com%`, `%insttrip.manulife.com%`). Four facts
make that worth re-deriving from evidence:

1. This table is **single-suite** — Adobe pins the report suite in feed config, so
   there is no `rsid` column on the row. Scope is URL-only.
2. `page_url` is 0% blank, `visit_start_page_url` and `first_hit_page_url` are ~17%
   blank, and `post_page_url` is 58.9% blank. The shipped GWAM pipeline default
   (post_page_url first) would drop ~60% of CoverMe rows.
3. Language is split by **domain**, not path: coverme.com=EN,
   pourmeproteger.com=FR, with insttrip.manulife.com as an English affinity product.
4. The tail includes non-trivial UAT (`uat.coverme.com` ~2.4%), Salesforce CMS
   (`mlc--cms.*.visual.force.com` ~0.6%), and AEM authoring (`author-aem-prod` ~0.1%)
   — each needs a business decision (include / exclude).

**Filters are TAGGED, never applied.** Today's scope patterns become boolean columns
so you can measure what they capture *and what they miss*. Noise (AEM authoring,
UAT, staging, dev) is likewise tagged, not dropped.

**Data visibility (ADR-0007 §5).** Hosts and paths print in full — that is the
deliverable. Query strings and fragments are stripped before grouping, since session
tokens live there and they would also shatter the grouping. ID-like path segments
generalize to `{id}`, which is what makes the rollup readable. No visitor, IP,
cookie, or user-agent column is projected at all — this notebook has no need for them.

**How to run.** Attach a cluster, run the S0 CONFIG cell to materialize widgets, then:
1. **Dry run first** — set `dry_run_month` to e.g. `2026-06` and `write_delta=false`.
   Confirms the whole notebook end-to-end for a fraction of the cost.
2. **Full run** — clear `dry_run_month`, set `target_catalog`, `write_delta=true`, Run All.

Each section prints a `===== BEGIN SHAREABLE: <id> =====` block. Sections are
independent: a failure prints `SKIPPED` and the run continues.

## S0 — Config, constants, helpers

In [0]:
import json
import re
import math
import datetime
import traceback

from pyspark.sql import functions as F

# ---------------------------------------------------------------- widgets ----
dbutils.widgets.text("table_fqn", "csdo_prod_catalog.adobe_coverme_bronze.hit_data", "1. Source table (catalog.schema.table)")
dbutils.widgets.text("start_date", "2023-01-01", "2. Scan start (YYYY-MM-DD)")
dbutils.widgets.text("end_date", "", "3. Scan end (YYYY-MM-DD, empty = today)")
dbutils.widgets.text("dry_run_month", "", "4. Dry run: single month YYYY-MM (empty = off)")
dbutils.widgets.text("target_catalog", "__SET_ME__", "5. Target catalog for Delta output")
dbutils.widgets.text("target_schema", "gmai_pulse_bronze", "6. Target schema for Delta output")
dbutils.widgets.text("write_delta", "true", "7. Write Delta tables (true/false)")
dbutils.widgets.text("min_hits", "1", "8. Drop inventory rows with fewer hits than this")
dbutils.widgets.text("top_n", "200", "9. Top-N cap for printed lists")
dbutils.widgets.text("prefix_depth", "5", "10. Path segments in the rollup prefix")

TABLE_FQN      = dbutils.widgets.get("table_fqn").strip()
START_DATE     = dbutils.widgets.get("start_date").strip()
END_DATE       = dbutils.widgets.get("end_date").strip()
DRY_RUN_MONTH  = dbutils.widgets.get("dry_run_month").strip()
TARGET_CATALOG = dbutils.widgets.get("target_catalog").strip()
TARGET_SCHEMA  = dbutils.widgets.get("target_schema").strip()
WRITE_DELTA    = dbutils.widgets.get("write_delta").strip().lower() == "true"
MIN_HITS       = int(dbutils.widgets.get("min_hits"))
TOP_N          = int(dbutils.widgets.get("top_n"))
# 5 segments is the meaningful default: CoverMe's deepest useful path
# (`/health-insurance/plan-selector/applicant-information`, etc.) sits at depth ~3-4,
# so depth 5 covers the leaf level with one buffer segment.
PREFIX_DEPTH   = max(1, int(dbutils.widgets.get("prefix_depth")))

PARTITION_COL = "hit_date"     # CoverMe partitions by hit_date (typed date)
RUN_DATE = datetime.date.today().isoformat()

# A dry run overrides the window with a single month — validate before the full scan.
if DRY_RUN_MONTH:
    _y, _m = (int(x) for x in DRY_RUN_MONTH.split("-"))
    START_DATE = datetime.date(_y, _m, 1).isoformat()
    _nm = datetime.date(_y + (_m == 12), (_m % 12) + 1, 1)
    END_DATE = (_nm - datetime.timedelta(days=1)).isoformat()
elif not END_DATE:
    END_DATE = datetime.date.today().isoformat()

# Writes are skipped unless a real catalog was supplied — the notebook stays fully
# runnable read-only, so a permissions gap degrades to print-only instead of failing.
CAN_WRITE = WRITE_DELTA and TARGET_CATALOG and TARGET_CATALOG != "__SET_ME__"

INVENTORY_FQN = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.coverme_url_scope_inventory"
ROLLUP_FQN    = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.coverme_url_scope_rollup"

# ------------------------------------------------------- scope constants ----
# Mirrors databricks/conf/settings.py — KEEP IN SYNC. These are TAGGED here, not
# applied; the point of this notebook is to measure what they capture and what they
# miss.
CUR_BROAD_LIKE = ["coverme.com", "pourmeproteger.com", "insttrip.manulife.com"]
CUR_EN_LIKE    = ["coverme.com", "insttrip.manulife.com"]
CUR_FR_LIKE    = ["pourmeproteger.com"]
NOISE_LIKE     = ["adobeaemcloud.com", "author-aem-prod.manulife.ca",
                  "uat.coverme.com", "uat.pourmeproteger.com",
                  "web-app.uat.affinitylife.aks.manulife.ca",
                  "tripx-web-en.uat.aks.manulife.ca", "tripx-web-fr.uat.aks.manulife.ca",
                  "coverme-web.uat", "www-aem-stage", "tripx.uat.coverme.com",
                  "coverme-en.uat.affinityhd", "localhost:5000"]

# Business-category regex (parity with coverme_eda.py S4c). Six product surface
# areas — health, travel, life, my-next-chapter (retirement content), vitality, home —
# plus their French equivalents (assurance-sante, assurance-voyage, assurance-vie).
CM_STRICT = (r"health-insurance|assurance-sante|travel-insurance|assurance-voyage"
             r"|life-insurance|assurance-vie|my-next-chapter|vitality"
             r"|/covme/health-insurance")
CM_BROAD  = r"insurance|assurance|coverme|covme|manulife|manuvie|pourmeproteger"

# URL candidates, in coalesce priority order. page_url leads because it is 0.001%
# blank while post_page_url is 58.9% blank on CoverMe — the exact OPPOSITE of the
# GWAM shipped pipeline default.
URL_CANDIDATES = ("page_url", "visit_start_page_url", "first_hit_page_url",
                  "post_page_url", "site_url")

# --------------------------------------------------------------- helpers ----
# Vendored verbatim from coverme_discovery_probe.py (S0). Keep in sync so the
# SHAREABLE protocol stays identical across CoverMe notebooks.
RESULTS = {}   # section_id -> payload
SKIPPED = {}   # section_id -> reason

MAX_EMIT_STR = 2000


def _scrub_str(s):
    if len(s) > MAX_EMIT_STR:
        s = s[:MAX_EMIT_STR] + "...<trunc>"
    return s


def _scrub(obj):
    """Walk a payload: truncate strings, round floats."""
    if isinstance(obj, dict):
        return {(_scrub_str(k) if isinstance(k, str) else k): _scrub(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_scrub(v) for v in obj]
    if isinstance(obj, str):
        return _scrub_str(obj)
    if isinstance(obj, float):
        return round(obj, 4) if math.isfinite(obj) else None
    return obj


def emit(section_id, payload):
    """Single output chokepoint: every shareable output goes through here."""
    payload = _scrub(payload)
    RESULTS[section_id] = payload
    body = json.dumps(payload, separators=(",", ":"), default=str)
    print(f"===== BEGIN SHAREABLE: {section_id} =====")
    if len(body) <= 48000:
        print(body)
    else:
        n_parts = math.ceil(len(body) / 40000)
        for i in range(n_parts):
            print(f"----- part {i+1} of {n_parts} (concatenate parts to reassemble) -----")
            print(body[i * 40000:(i + 1) * 40000])
    print(f"===== END SHAREABLE: {section_id} =====")


def run_section(section_id, fn):
    print(f"\n>>> running {section_id} ...")
    t0 = datetime.datetime.now()
    try:
        fn()
        print(f">>> {section_id} done in {(datetime.datetime.now() - t0).total_seconds():.0f}s")
    except Exception as e:
        reason = f"{type(e).__name__}: {str(e)[:300]}"
        SKIPPED[section_id] = reason
        print(f"===== SKIPPED: {section_id} | {reason} =====")
        traceback.print_exc()


def qcol(col_name):
    """F.col with backtick quoting — Adobe schemas can carry dotted column names."""
    return F.col("`" + col_name.replace("`", "``") + "`")


def nonblank(col_name):
    """Adobe feeds use empty strings, not NULLs."""
    c = qcol(col_name)
    return c.isNotNull() & (F.trim(c.cast("string")) != "")


def pick_col(df, *candidates):
    """First candidate column present in the schema, else None."""
    cols = set(df.columns)
    for c in candidates:
        if c in cols:
            return c
    return None


def pct(num, den):
    return round(100.0 * num / den, 4) if den else None


def coverage(rows_total, rows_emitted, hits_emitted, hits_total):
    """Truncation is always visible — never a silent top-N."""
    return {"rows_total": rows_total, "rows_emitted": rows_emitted,
            "truncated": rows_total > rows_emitted,
            "hits_pct_covered_by_emitted": pct(hits_emitted, hits_total)}


print(f"window: {START_DATE} .. {END_DATE}" + (f"  (DRY RUN {DRY_RUN_MONTH})" if DRY_RUN_MONTH else ""))
print(f"table : {TABLE_FQN}")
print(f"write : {'YES -> ' + INVENTORY_FQN if CAN_WRITE else 'NO (print-only)'}")

window: 2023-01-01 .. 2026-07-22
table : csdo_prod_catalog.adobe_coverme_bronze.hit_data
write : NO (print-only)


## S1 — Schema probe (no scan)
Confirms the table resolves, which URL candidates actually exist, and that the
partition column is present. Metadata only.

In [0]:
SRC = spark.table(TABLE_FQN)

RSID_COL = pick_col(SRC, "rsid", "report_suite", "reportsuite", "reportsuiteid", "post_rsid")
PRESENT_URL_COLS = [c for c in URL_CANDIDATES if c in set(SRC.columns)]
COAL_ORDER = [c for c in URL_CANDIDATES if c in PRESENT_URL_COLS]

if not PRESENT_URL_COLS:
    raise ValueError(f"None of {URL_CANDIDATES} exist in {TABLE_FQN} — nothing to inventory.")
if PARTITION_COL not in SRC.columns:
    raise ValueError(f"Partition column {PARTITION_COL} missing — refusing an unpruned full scan.")

PART_TYPE = dict(SRC.dtypes)[PARTITION_COL]
SINGLE_SUITE = RSID_COL is None


def _part_pred():
    """Predicate honoring the real partition dtype so Delta prunes partitions.
    A string 'YYYY-MM-DD' partition compares correctly lexically, so one predicate
    covers both dtypes."""
    lo, hi = F.lit(START_DATE), F.lit(END_DATE)
    if PART_TYPE == "date":
        lo, hi = lo.cast("date"), hi.cast("date")
    return (F.col(PARTITION_COL) >= lo) & (F.col(PARTITION_COL) <= hi)


run_section("s1_schema_probe", lambda: emit("s1_schema_probe", {
    "table": TABLE_FQN,
    "rsid_col": RSID_COL,
    "single_suite": SINGLE_SUITE,
    "partition_col": PARTITION_COL,
    "partition_dtype": PART_TYPE,
    "url_candidates_declared": list(URL_CANDIDATES),
    "url_candidates_present": PRESENT_URL_COLS,
    "url_candidates_missing": [c for c in URL_CANDIDATES if c not in PRESENT_URL_COLS],
    "coalesce_order": COAL_ORDER,
    "window": {"start": START_DATE, "end": END_DATE, "dry_run_month": DRY_RUN_MONTH or None},
    "reading_note": ("single_suite=true is expected for csdo_prod_catalog.adobe_coverme_bronze — "
                     "Adobe pins the report suite in feed config, so there is no rsid column on "
                     "the row. Scope is URL-only."),
}))


>>> running s1_schema_probe ...
===== BEGIN SHAREABLE: s1_schema_probe =====
{"table":"csdo_prod_catalog.adobe_coverme_bronze.hit_data","rsid_col":null,"single_suite":true,"partition_col":"hit_date","partition_dtype":"date","url_candidates_declared":["page_url","visit_start_page_url","first_hit_page_url","post_page_url","site_url"],"url_candidates_present":["page_url","visit_start_page_url","first_hit_page_url","post_page_url"],"url_candidates_missing":["site_url"],"coalesce_order":["page_url","visit_start_page_url","first_hit_page_url","post_page_url"],"window":{"start":"2023-01-01","end":"2026-07-22","dry_run_month":null},"reading_note":"single_suite=true is expected for csdo_prod_catalog.adobe_coverme_bronze \u2014 Adobe pins the report suite in feed config, so there is no rsid column on the row. Scope is URL-only."}
===== END SHAREABLE: s1_schema_probe =====
>>> s1_schema_probe done in 0s


## S2 — Host presence probe (cheap: 2 columns)
The CoverMe analog of the GWAM `suite_presence` probe. Projects only `hit_date` +
the coalesced URL host, so columnar storage never reads the wide URL strings.
Answers "does `pourmeproteger.com` exist in this window at all?" **before** S3 spends
real compute — and confirms whether all three broad-scope hosts (`coverme.com`,
`pourmeproteger.com`, `insttrip.manulife.com`) are populated.

In [0]:
def _s2():
    # Blank-guarded coalesce (page_url first — see COAL_ORDER). Adobe writes empty
    # strings, not NULLs, so blanks must be mapped to NULL before the coalesce.
    _complete = F.coalesce(*[F.when(nonblank(c), qcol(c)) for c in COAL_ORDER])
    _u  = F.lower(F.trim(_complete.cast("string")))
    _u  = F.regexp_replace(_u, r"^[a-z]+://", "")
    _u  = F.regexp_replace(_u, r"^www\.", "")
    _host = F.regexp_extract(_u, r"^([^/?#]+)", 1)

    probe = (SRC
             .filter(_part_pred())
             .select(_host.alias("host"), F.col(PARTITION_COL).alias("pd"))
             .filter(F.col("host") != F.lit("")))

    # For each of the three broad-scope hosts, is it present in the window?
    per_host = {}
    for _h in CUR_BROAD_LIKE:
        r = probe.filter(F.col("host").contains(_h)).agg(
            F.count("*").alias("hits"),
            F.countDistinct("pd").alias("active_days"),
            F.min("pd").cast("string").alias("first_day"),
            F.max("pd").cast("string").alias("last_day"),
        ).collect()[0].asDict()
        per_host[_h] = r if (r["hits"] or 0) > 0 else {"hits": 0, "note": "ABSENT in this window"}

    monthly = (probe
               .groupBy(F.substring(F.col("pd").cast("string"), 1, 7).alias("month"))
               .agg(F.count("*").alias("hits"))
               .orderBy("month")
               .collect())

    emit("s2_host_presence", {
        "per_broad_host": per_host,
        "monthly_hits_total": [{"month": r["month"], "hits": r["hits"]} for r in monthly],
        "reading_note": ("per_broad_host reports each of the three CoverMe production hosts "
                         "(coverme.com, pourmeproteger.com, insttrip.manulife.com) with hit "
                         "count + date range. A host at 0 means the broad-scope filter would "
                         "silently be single-language / single-brand."),
    })


run_section("s2_host_presence", _s2)


>>> running s2_host_presence ...
===== BEGIN SHAREABLE: s2_host_presence =====
{"per_broad_host":{"coverme.com":{"hits":51787657,"active_days":1210,"first_day":"2023-02-28","last_day":"2026-07-21"},"pourmeproteger.com":{"hits":5824708,"active_days":1210,"first_day":"2023-02-28","last_day":"2026-07-21"},"insttrip.manulife.com":{"hits":1487628,"active_days":260,"first_day":"2023-02-28","last_day":"2024-03-11"}},"monthly_hits_total":[{"month":"2023-02","hits":41717},{"month":"2023-03","hits":1099189},{"month":"2023-04","hits":894858},{"month":"2023-05","hits":930259},{"month":"2023-06","hits":943387},{"month":"2023-07","hits":923514},{"month":"2023-08","hits":916704},{"month":"2023-09","hits":807998},{"month":"2023-10","hits":1151459},{"month":"2023-11","hits":1338547},{"month":"2023-12","hits":1696317},{"month":"2024-01","hits":2203748},{"month":"2024-02","hits":1984099},{"month":"2024-03","hits":2272225},{"month":"2024-04","hits":2185868},{"month":"2024-05","hits":2388940},{"month":"20

## S3 — Build the inventory (the one expensive scan)
Partition prune → projection prune → normalize → aggregate.
No URL predicate. No `collect()` of raw rows.

In [0]:
BASE = (SRC
        .filter(_part_pred())                              # 1. partition pruning FIRST
        .select(F.col(PARTITION_COL).alias("pd"),           # 2. projection pruning
                *[qcol(c).alias(c) for c in PRESENT_URL_COLS]))

# --- normalization -------------------------------------------------------------
# Blank-guarded coalesce: Adobe writes empty strings, not NULLs, so a plain coalesce
# would happily return "" from page_url and never reach the next column. On CoverMe
# this rarely matters (page_url is 0.001% blank), but the guard is kept for parity.
_complete = F.coalesce(*[F.when(nonblank(c), qcol(c)) for c in COAL_ORDER])

_u  = F.lower(F.trim(_complete.cast("string")))
_u  = F.regexp_replace(_u, r"^[a-z]+://", "")              # drop scheme
_u  = F.regexp_replace(_u, r"^www\.", "")                  # drop www.
_hp = F.regexp_extract(_u, r"^([^?#]*)", 1)                # drop query + fragment (PII)
_hp = F.regexp_replace(_hp, r"/+$", "")                    # drop trailing slash

_host = F.regexp_extract(_hp, r"^([^/]+)", 1)
_path = F.when(_hp.rlike(r"^[^/]+/"),
               F.concat(F.lit("/"), F.regexp_extract(_hp, r"^[^/]+/(.*)$", 1))
               ).otherwise(F.lit("/"))

# Rows where every URL candidate was blank get their own bucket rather than being
# dropped, so the blank share stays visible in the output.
_is_blank = _hp.isNull() | (_hp == "")
HOST = F.when(_is_blank, F.lit("<blank>")).otherwise(_host)
PATH = F.when(_is_blank, F.lit("<blank>")).otherwise(_path)

# ID generalization: collapses /quote/12345 into /quote/{id}. Both a privacy control
# and what makes the rollup readable — otherwise leaf IDs explode into singleton rows.
_pg = PATH
_pg = F.regexp_replace(_pg, r"/[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}(?=/|$)", "/{id}")
_pg = F.regexp_replace(_pg, r"/[0-9a-f]{16,}(?=/|$)", "/{id}")
_pg = F.regexp_replace(_pg, r"/\d{4,}(?=/|$)", "/{id}")
PATH_GEN = _pg


def _seg(n):
    """n-th path segment of the generalized path ('' when absent)."""
    return F.regexp_extract(PATH_GEN, r"^" + ("/[^/]*" * (n - 1)) + r"/([^/]*)", 1)


_segs = [_seg(n) for n in range(1, PREFIX_DEPTH + 1)]
PATH_PREFIX = F.concat(
    F.lit("/"), _segs[0],
    *[F.when(s != "", F.concat(F.lit("/"), s)).otherwise(F.lit("")) for s in _segs[1:]],
)


# --- tags (measured, never applied as filters) ---------------------------------
def _any_contains(col, needles):
    cond = F.lit(False)
    for n in needles:
        cond = cond | col.contains(n)
    return cond


TAGS = {
    # Language tag: coverme.com=EN, pourmeproteger.com=FR, plus path fallback.
    "lang": (F.when(HOST.rlike(r"pourmeproteger|manuvie|assurance-manuvie"), "fr")
              .when(HOST.rlike(r"coverme\.com|insttrip\.manulife\.com"), "en")
              .when(PATH_GEN.rlike(r"regimes-collectif|retraite|particuliers|assurance-|blogue|conseillers"), "fr")
              .when(PATH_GEN.rlike(r"group-retirement|group-plans|blog|health-insurance|travel-insurance|life-insurance"), "en")
              .otherwise("unknown")),
    "matches_current_broad": _any_contains(_hp, CUR_BROAD_LIKE),
    "matches_current_en":    _any_contains(_hp, CUR_EN_LIKE),
    "matches_current_fr":    _any_contains(_hp, CUR_FR_LIKE),
    "is_noise":              _any_contains(_hp, NOISE_LIKE),
    "cm_strict":             _hp.rlike(CM_STRICT),
    "cm_broad":              _hp.rlike(CM_BROAD),
    "env": (F.when(HOST.contains("adobeaemcloud.com") | HOST.contains("author-aem"), "aem")
             .when(HOST.rlike(r"preview|stage|staging|author|(^|\.)uat($|\.)|\.uat\.|(^|\.)dev\."), "nonprod")
             .otherwise("prod")),
}

GRAIN = ["host", "path", "path_prefix"] + list(TAGS.keys())

NORM = BASE.select(
    F.col("pd"),
    HOST.alias("host"), PATH.alias("path"), PATH_PREFIX.alias("path_prefix"),
    *[expr.alias(name) for name, expr in TAGS.items()],
    # Per-source-column population, carried through the same scan so blank rates come
    # free instead of costing a second pass over the source table.
    *[F.when(nonblank(c), 1).otherwise(0).alias(f"src_{c}_nb") for c in PRESENT_URL_COLS],
)

INV = (NORM.groupBy(*GRAIN)
       .agg(F.count("*").alias("hits"),
            F.countDistinct("pd").alias("active_days"),
            F.min("pd").cast("string").alias("first_seen"),
            F.max("pd").cast("string").alias("last_seen"),
            *[F.sum(f"src_{c}_nb").alias(f"src_{c}_nb") for c in PRESENT_URL_COLS])
       .filter(F.col("hits") >= F.lit(MIN_HITS))
       .withColumn("run_date", F.lit(RUN_DATE)))

# The rollup is derived from INV (already small), so caching INV once turns the
# rollup, the summary, and both writes into a single pass over the source table.
INV = INV.persist()
INV_ROWS = INV.count()
print(f"inventory rows: {INV_ROWS:,}  (min_hits={MIN_HITS})")

ROLLUP = (INV.groupBy("host", "path_prefix", "lang", "env",
                      "matches_current_broad", "matches_current_en", "matches_current_fr",
                      "is_noise", "cm_strict", "cm_broad")
          .agg(F.sum("hits").alias("hits"),
               F.countDistinct("path").alias("distinct_paths"),
               F.min("first_seen").alias("first_seen"),
               F.max("last_seen").alias("last_seen"))
          .withColumn("run_date", F.lit(RUN_DATE)))

inventory rows: 3,859  (min_hits=1)


## S4 — Write Delta tables
Partitioned by `run_date` with `replaceWhere`, so re-running the same day is
idempotent while earlier runs are retained. Skipped entirely when `target_catalog`
is `__SET_ME__`. A schema change needs the table dropped first (`replaceWhere` and
`overwriteSchema` cannot be combined).

In [0]:
def _write(df, fqn):
    (df.write.format("delta")
       .mode("overwrite")
       .option("replaceWhere", f"run_date = '{RUN_DATE}'")
       .partitionBy("run_date")
       .saveAsTable(fqn))
    return spark.table(fqn).filter(F.col("run_date") == RUN_DATE).count()


def _s4():
    if not CAN_WRITE:
        emit("s4_delta_write", {"written": False,
                                "reason": "target_catalog is __SET_ME__ or write_delta=false",
                                "note": "notebook ran print-only; all figures below are still valid"})
        return
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_CATALOG}.{TARGET_SCHEMA}")
    n_inv = _write(INV, INVENTORY_FQN)
    n_rol = _write(ROLLUP, ROLLUP_FQN)
    emit("s4_delta_write", {"written": True, "run_date": RUN_DATE,
                            "inventory": {"table": INVENTORY_FQN, "rows": n_inv},
                            "rollup": {"table": ROLLUP_FQN, "rows": n_rol}})


run_section("s4_delta_write", _s4)


>>> running s4_delta_write ...
===== BEGIN SHAREABLE: s4_delta_write =====
{"written":false,"reason":"target_catalog is __SET_ME__ or write_delta=false","note":"notebook ran print-only; all figures below are still valid"}
===== END SHAREABLE: s4_delta_write =====
>>> s4_delta_write done in 0s


## S5 — Inventory summary + URL column availability
Blank % by source column is the headline check: `post_page_url` should read ~58.9%
blank on this table, while `page_url` reads ~0%. If those numbers reverse, the
recommended coalesce order in `coverme_eda.py` (page_url first) needs to be revisited.

In [0]:
def _s5():
    agg = INV.agg(
        F.sum("hits").alias("hits"),
        F.count("*").alias("distinct_host_path"),
        F.countDistinct("host").alias("distinct_hosts"),
        F.min("first_seen").alias("first_seen"),
        F.max("last_seen").alias("last_seen"),
        F.sum(F.when(F.col("host") == "<blank>", F.col("hits")).otherwise(0)).alias("blank_url_hits"),
        *[F.sum(f"src_{c}_nb").alias(f"src_{c}_nb") for c in PRESENT_URL_COLS],
    ).collect()[0].asDict()
    hits = agg["hits"] or 0

    summary = {
        "hits": hits,
        "distinct_host_path": agg["distinct_host_path"],
        "distinct_hosts": agg["distinct_hosts"],
        "date_range": [agg["first_seen"], agg["last_seen"]],
        "blank_url_hits": agg["blank_url_hits"],
        "blank_url_pct": pct(agg["blank_url_hits"], hits),
        # The headline check: post_page_url should read ~58.9% blank on CoverMe, while
        # page_url reads ~0%. If those numbers reverse, EDA S4a should be re-run.
        "blank_pct_by_source_col": {c: pct(hits - agg[f"src_{c}_nb"], hits) for c in PRESENT_URL_COLS},
    }

    hosts = (INV.groupBy("host").agg(F.sum("hits").alias("hits"))
             .orderBy(F.col("hits").desc()).limit(TOP_N).collect())

    emit("s5_inventory_summary", {
        "summary": summary,
        "top_hosts": [{"host": r["host"], "hits": r["hits"]} for r in hosts],
        "top_hosts_coverage": coverage(INV.select("host").distinct().count(),
                                       len(hosts), sum(r["hits"] for r in hosts), hits),
    })


run_section("s5_inventory_summary", _s5)


>>> running s5_inventory_summary ...
===== BEGIN SHAREABLE: s5_inventory_summary =====
{"summary":{"hits":60102247,"distinct_host_path":3859,"distinct_hosts":311,"date_range":["2023-02-28","2026-07-21"],"blank_url_hits":271,"blank_url_pct":0.0005,"blank_pct_by_source_col":{"page_url":0.0005,"visit_start_page_url":16.7367,"first_hit_page_url":16.872,"post_page_url":58.8962}},"top_hosts":[{"host":"coverme.com","hits":50370229},{"host":"pourmeproteger.com","hits":5818684},{"host":"insttrip.manulife.com","hits":1487628},{"host":"uat.coverme.com","hits":1416761},{"host":"web-app.uat.affinitylife.aks.manulife.ca","hits":439371},{"host":"mlc--cms.na154.visual.force.com","hits":210577},{"host":"mlc--cms.can52.visual.sfdc-58ktaz.force.com","hits":135735},{"host":"coverme-en.apps.cac.pcf.manulife.com","hits":62516},{"host":"author-aem-prod.manulife.ca","hits":52345},{"host":"web-app.prod.affinitylife.aks.manulife.ca","hits":28772},{"host":"manulife-insurance.ca","hits":20101},{"host":"tripx-web

## S6 — Path-prefix rollup
The scannable view: host + first `prefix_depth` path segments, IDs generalized.

In [0]:
def _s6():
    total_hits = ROLLUP.agg(F.sum("hits")).collect()[0][0] or 0
    rows_total = ROLLUP.count()
    top = ROLLUP.orderBy(F.col("hits").desc()).limit(TOP_N).collect()
    emit("s6_url_scope_rollup", {
        "rows": [{"host": r["host"], "path_prefix": r["path_prefix"],
                  "hits": r["hits"], "distinct_paths": r["distinct_paths"],
                  "lang": r["lang"], "env": r["env"],
                  "first_seen": r["first_seen"], "last_seen": r["last_seen"],
                  "broad": r["matches_current_broad"], "en": r["matches_current_en"],
                  "fr": r["matches_current_fr"], "noise": r["is_noise"]}
                 for r in top],
        "coverage": coverage(rows_total, len(top), sum(r["hits"] for r in top), total_hits),
    })


run_section("s6_url_scope_rollup", _s6)


>>> running s6_url_scope_rollup ...
===== BEGIN SHAREABLE: s6_url_scope_rollup =====
----- part 1 of 2 (concatenate parts to reassemble) -----
{"rows":[{"host":"coverme.com","path_prefix":"/covme/health-insurance/quote","hits":5063072,"distinct_paths":1,"lang":"en","env":"prod","first_seen":"2023-02-28","last_seen":"2026-07-21","broad":true,"en":true,"fr":false,"noise":false},{"host":"coverme.com","path_prefix":"/travel-insurance/get-a-quote","hits":5062946,"distinct_paths":1,"lang":"en","env":"prod","first_seen":"2023-09-25","last_seen":"2026-07-21","broad":true,"en":true,"fr":false,"noise":false},{"host":"coverme.com","path_prefix":"/","hits":4461569,"distinct_paths":1,"lang":"en","env":"prod","first_seen":"2023-02-28","last_seen":"2026-07-21","broad":true,"en":true,"fr":false,"noise":false},{"host":"coverme.com","path_prefix":"/health-insurance.html","hits":3777637,"distinct_paths":1,"lang":"en","env":"prod","first_seen":"2023-02-28","last_seen":"2026-07-21","broad":true,"en":true,

## S7 — Scope filter comparison
**The decision section.** For each shipped filter (broad / en / fr): what it captures,
and — the number that matters — what falls outside *every* production filter but
still looks like CoverMe traffic.

In [0]:
def _s7():
    flags = ["matches_current_broad", "matches_current_en", "matches_current_fr",
             "is_noise", "cm_strict", "cm_broad"]
    # "uncovered" = matches the CoverMe regex (strict), is not noise, and is captured
    # by NEITHER the current broad filter (which includes both en + fr surfaces).
    uncovered_cond = (F.col("cm_strict") & ~F.col("is_noise") & ~F.col("matches_current_broad"))

    agg = INV.agg(
        F.sum("hits").alias("hits"),
        F.count("*").alias("urls"),
        *[F.sum(F.when(F.col(f), F.col("hits")).otherwise(0)).alias(f"{f}_hits") for f in flags],
        *[F.sum(F.when(F.col(f), 1).otherwise(0)).alias(f"{f}_urls") for f in flags],
        F.sum(F.when(uncovered_cond, F.col("hits")).otherwise(0)).alias("uncovered_cm_hits"),
        F.sum(F.when(uncovered_cond, 1).otherwise(0)).alias("uncovered_cm_urls"),
    ).collect()[0]

    d = agg.asDict()
    per_filter = {}
    for f in flags:
        per_filter[f] = {"hits": d[f"{f}_hits"], "urls": d[f"{f}_urls"],
                         "hits_pct": pct(d[f"{f}_hits"], d["hits"])}

    uncovered = (ROLLUP.filter(uncovered_cond)
                 .orderBy(F.col("hits").desc()).limit(TOP_N).collect())

    emit("s7_scope_filter_comparison", {
        "filters_evaluated": {
            "current_broad": CUR_BROAD_LIKE,
            "current_en":    CUR_EN_LIKE,
            "current_fr":    CUR_FR_LIKE,
            "noise_excluded_by_pipeline": NOISE_LIKE,
        },
        "total_hits": d["hits"],
        "total_urls": d["urls"],
        "by_filter": per_filter,
        "uncovered_coverme_hits": d["uncovered_cm_hits"],
        "uncovered_coverme_urls": d["uncovered_cm_urls"],
        "uncovered_coverme_pct": pct(d["uncovered_cm_hits"], d["hits"]),
        "top_uncovered_coverme_prefixes": [
            {"host": r["host"], "path_prefix": r["path_prefix"],
             "hits": r["hits"], "lang": r["lang"], "env": r["env"],
             "first_seen": r["first_seen"], "last_seen": r["last_seen"]} for r in uncovered],
        "reading_note": ("uncovered = matches the CoverMe strict regex (health / travel / life / "
                         "my-next-chapter / vitality / assurance-*), is not noise, and is "
                         "captured by the current broad filter. These are the candidate additions "
                         "to scope — likely alternate CoverMe domains or forgotten insurance "
                         "paths on manulife.com."),
    })


run_section("s7_scope_filter_comparison", _s7)


>>> running s7_scope_filter_comparison ...
===== BEGIN SHAREABLE: s7_scope_filter_comparison =====
{"filters_evaluated":{"current_broad":["coverme.com","pourmeproteger.com","insttrip.manulife.com"],"current_en":["coverme.com","insttrip.manulife.com"],"current_fr":["pourmeproteger.com"],"noise_excluded_by_pipeline":["adobeaemcloud.com","author-aem-prod.manulife.ca","uat.coverme.com","uat.pourmeproteger.com","web-app.uat.affinitylife.aks.manulife.ca","tripx-web-en.uat.aks.manulife.ca","tripx-web-fr.uat.aks.manulife.ca","coverme-web.uat","www-aem-stage","tripx.uat.coverme.com","coverme-en.uat.affinityhd","localhost:5000"]},"total_hits":60102247,"total_urls":3859,"by_filter":{"matches_current_broad":{"hits":59100004,"urls":1764,"hits_pct":98.3324},"matches_current_en":{"hits":53275295,"urls":1222,"hits_pct":88.6411},"matches_current_fr":{"hits":5824709,"urls":542,"hits_pct":9.6913},"is_noise":{"hits":1930059,"urls":966,"hits_pct":3.2113},"cm_strict":{"hits":49485692,"urls":1652,"hits_pct"

## S8 — Run manifest

In [0]:
run_section("s8_run_manifest", lambda: emit("s8_run_manifest", {
    "run_date": RUN_DATE,
    "generated_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "source_table": TABLE_FQN,
    "window": {"start": START_DATE, "end": END_DATE, "dry_run_month": DRY_RUN_MONTH or None},
    "resolved": {"rsid_col": RSID_COL, "single_suite": SINGLE_SUITE,
                 "url_cols_present": PRESENT_URL_COLS,
                 "coalesce_order": COAL_ORDER, "partition_col": PARTITION_COL},
    "min_hits": MIN_HITS, "top_n": TOP_N, "prefix_depth": PREFIX_DEPTH,
    "inventory_rows": INV_ROWS,
    "delta_written": CAN_WRITE,
    "tables": {"inventory": INVENTORY_FQN if CAN_WRITE else None,
               "rollup": ROLLUP_FQN if CAN_WRITE else None},
    "sections_emitted": sorted(RESULTS.keys()),
    "sections_skipped": SKIPPED,
}))

INV.unpersist()


>>> running s8_run_manifest ...
===== BEGIN SHAREABLE: s8_run_manifest =====
{"run_date":"2026-07-22","generated_at":"2026-07-22T20:10:35","source_table":"csdo_prod_catalog.adobe_coverme_bronze.hit_data","window":{"start":"2023-01-01","end":"2026-07-22","dry_run_month":null},"resolved":{"rsid_col":null,"single_suite":true,"url_cols_present":["page_url","visit_start_page_url","first_hit_page_url","post_page_url"],"coalesce_order":["page_url","visit_start_page_url","first_hit_page_url","post_page_url"],"partition_col":"hit_date"},"min_hits":1,"top_n":200,"prefix_depth":5,"inventory_rows":3859,"delta_written":false,"tables":{"inventory":null,"rollup":null},"sections_emitted":["s1_schema_probe","s2_host_presence","s4_delta_write","s5_inventory_summary","s6_url_scope_rollup","s7_scope_filter_comparison"],"sections_skipped":{}}
===== END SHAREABLE: s8_run_manifest =====
>>> s8_run_manifest done in 0s


DataFrame[host: string, path: string, path_prefix: string, lang: string, matches_current_broad: boolean, matches_current_en: boolean, matches_current_fr: boolean, is_noise: boolean, cm_strict: boolean, cm_broad: boolean, env: string, hits: bigint, active_days: bigint, first_seen: string, last_seen: string, src_page_url_nb: bigint, src_visit_start_page_url_nb: bigint, src_first_hit_page_url_nb: bigint, src_post_page_url_nb: bigint, run_date: string]